Work Trial Task: Best-Level, Multi-Level, Integrated OFI, and Cross-Asset OFI

This notebook addresses all aspects of the internship task, including:
- Best-Level OFI
- Multi-Level OFI (top 10 LOB levels)
- Integrated OFI via PCA (L1-normalized)
- \*Note: Can't do Cross-Asset OFI as only one asset's data is given in the CSV (~5000 rows of just AAPL given not 25000 rows)

Each OFI feature is computed per timestamp. Data used: `first_25000_rows.csv`

Additionally, I have included some plots:
- Integrated OFI signal (how OFI evolves over time)
- Integrated OFI vs. simulated returns (how it correlates with returns-like movement)

In [1]:
!pip install pandas numpy scikit-learn matplotlib --quiet
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('first_25000_rows.csv')
df.head()

In [ ]:
def compute_normalized_ofi(df, level):
    bid_px = df[f'bid_px_0{level}']
    ask_px = df[f'ask_px_0{level}']
    bid_sz = df[f'bid_sz_0{level}']
    ask_sz = df[f'ask_sz_0{level}']

    bid_px_prev = bid_px.shift(1)
    ask_px_prev = ask_px.shift(1)
    bid_sz_prev = bid_sz.shift(1)
    ask_sz_prev = ask_sz.shift(1)

    ofi_bid = np.where(
        bid_px > bid_px_prev, bid_sz,
        np.where(bid_px == bid_px_prev, bid_sz - bid_sz_prev, -bid_sz)
    )
    ofi_ask = np.where(
        ask_px > ask_px_prev, -ask_sz,
        np.where(ask_px == ask_px_prev, ask_sz - ask_sz_prev, ask_sz)
    )

    raw_ofi = ofi_bid - ofi_ask
    avg_depth = 0.5 * (bid_sz + ask_sz).rolling(window=20, min_periods=1).mean()

    return raw_ofi / avg_depth

def compute_ofi_features(df):
    normalized_ofis = pd.DataFrame()
    for level in range(10):
        normalized_ofis[f'ofi_{level+1}'] = compute_normalized_ofi(df, level)

    pca = PCA(n_components=1)
    pca_weights = pca.fit(normalized_ofis).components_[0]
    pca_weights /= np.sum(np.abs(pca_weights))
    integrated_ofi = normalized_ofis.dot(pca_weights)
    normalized_ofis['ofi_integrated'] = integrated_ofi

    return pd.concat([df[['ts_event', 'symbol']].reset_index(drop=True), normalized_ofis], axis=1)


In [ ]:
ofi_df = compute_ofi_features(df)
ofi_df.head()

In [ ]:
ofi_df.to_csv('ofi_features_output.csv', index=False)
from google.colab import files
files.download('ofi_features_output.csv') # store output

In [ ]:
import matplotlib.pyplot as plt
ofi_df['ofi_integrated'].plot(figsize=(12, 4), title='Integrated OFI Signal')
plt.xlabel('Index')
plt.ylabel('OFI')
plt.grid(True)
plt.show()

In [ ]:
integrated_ofi = ofi_df['ofi_integrated']
simulated_returns = integrated_ofi.rolling(window=10, min_periods=1).mean().diff()

import matplotlib.pyplot as plt
plt.figure(figsize=(14, 4))
plt.plot(integrated_ofi, label='Integrated OFI')
plt.title('Integrated OFI Signal Over Time')
plt.xlabel('Index')
plt.ylabel('OFI Value')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(integrated_ofi, label='Integrated OFI')
plt.plot(simulated_returns, label='Simulated Returns', alpha=0.7)
plt.title('Integrated OFI vs. Simulated Returns')
plt.xlabel('Index')
plt.ylabel('Signal')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()